### 1. Выражения, сорта и декларации
#### Задача 1.

Изучение функций исследования дерева.

In [1]:
from z3 import *

a = Int('a')
b = Real('b')
c = FP("c", Float64())
d = BitVec("d", 64)

print (a.sort()) # сорт Int
print (b.sort()) # сорт Real
print (c.sort()) # сорт FPSort(11, 53)
print (d.sort()) # сорт BitVec(64)
print ((b == b).sort()) # сорт Bool

Int
Real
FPSort(11, 53)
BitVec(64)
Bool


In [2]:
x = Int('x')

expr1 = x ** 2 + 5
expr2 =  5 + x ** 2

print (eq(expr1, expr2)) # False
print (eq(expr1, expr1)) # True

False
True


In [3]:
print(expr1.hash())
print(expr2.hash())
print(expr1.hash() == expr2.hash())

2673758837
626320335
False


In [4]:
expr_decl = expr1.decl()

print ("expr1 is expression: ", is_expr(expr1))
print ("expr_decl: ", expr_decl)
print ("expr_decl name: ", expr_decl.name())
print ("expr_decl range: ", expr_decl.range())
print ("expr_decl arity: ", expr_decl.arity())

for i in range(expr_decl.arity()):
    print ("domain(", i, "): ", expr_decl.domain(i)) # Int, Real

expr1 is expression:  True
expr_decl:  +
expr_decl name:  +
expr_decl range:  Real
expr_decl arity:  2
domain( 0 ):  Real
domain( 1 ):  Real


#### Задача 2. substitute

In [5]:
expr1 = x + 1
expr2 = 3 * (x + 1) + (x + 150) * (x + 1) - 1

print("substitute: ", substitute(expr2, (expr1, IntVal(5))))

substitute:  3*5 + (x + 150)*5 - 1


или запишем формулу расчёта пути при равноускоренном движении где ускорение выражено через скорость?

In [6]:
s, v0, v, a, t = Reals("s v0 v a t")

distance = v0 * t + (((v - v0) / t) * t**2 ) / 2

acc = (v - v0) / t

result = substitute(distance, 
                    (acc, a), 
                    (v0 * t, RealVal(0))
                   )

print(result, " -> ", simplify(result))


0 + (a*t**2)/2  ->  1/2*a*t**2


### 2. Массивы.

#### Задача 1. Доказать что массив не строго отсортирован по возрастанию.

In [7]:
import random

arr_len = 10
arr = Array('arr', IntSort(), IntSort())

for i in range(arr_len):
    arr = Store(arr, i, IntVal(i + 1))

for i in range(arr_len):
    expr = Select(arr, i)
    print(f'[{i}] = ', simplify(expr))

[0] =  1
[1] =  2
[2] =  3
[3] =  4
[4] =  5
[5] =  6
[6] =  7
[7] =  8
[8] =  9
[9] =  10


In [8]:
i = Int('i')
s = Solver()
s.add(And(0 <= i, i < arr_len))
s.add(ForAll(i, arr[i] < arr[i - 1]))

s.check()

unsat

unsat подтверждает наши утверждения. Проверим прувером:

In [9]:
prove(ForAll([i], 
             Implies(
                     And(i > 0, i < arr_len),
                     arr[i] >= arr[i - 1]
                    )
            )
      )

proved


#### Задача 2. Доказать, что при изменении массива создаётся новый с измененным значением.

In [10]:
arr = Array('arr', IntSort(), IntSort())

number = IntVal(300000)
index = IntVal(5)

new_arr = Store(arr, index, number)

prove(Implies(arr[index] != number, And(arr != new_arr, new_arr[index] == number)))

proved


### 3. Кванторы. 

#### Задача 1. Транзитивность через кванторы.

In [11]:
x, y, z = Ints('x y z')

prove(ForAll([x, y, z], 
             Implies(And(x < y, y < z), x < z)))

proved


#### Задача 2. Каждый сотрудник работает ровно в одном отделе.

In [12]:
employee = DeclareSort('employee')
department = DeclareSort('department')

is_work_in_department = Function('is_work_in_department', employee, department, BoolSort())

emp = Const('emp', employee)
dep1, dep2 = Consts('dep1 dep2', department)

s = Solver()

expr = ForAll([emp, dep1, dep2], 
              Implies(And(is_work_in_department(emp, dep1),
                          is_work_in_department(emp, dep2)),
                      dep1 == dep2
                     )
             )

s.add(expr)

viktor = Const('viktor', employee)
it = Const('it', department)
hr = Const('hr', department)

s.add(is_work_in_department(viktor, it))
s.add(is_work_in_department(viktor, hr))
s.add(it != hr)

if s.check() == unsat:
    print('proved')
else:
    print('has counterexample')

proved


### 4. Оптимизация.

#### Задача 1. Максимальная прибыль предприятия.

Предприятие выпускает два вида продукции - изделие p1 и изделие p2. Для производства используются два станка b1 и b2.

Каждое изделие p1 занимает:
- 2 часа работы станка b1;
- 1 час работы станка b2;

Каждое изделие p2 занимает:
- 1 час работы станка b1;
- 3 часа работы станка b2;

За смену доступно:
- не более 100 часов работы станка b1;
- не более 120 часов работы станка b2;

Прибыль от продажи одной единицы продукции составляет:
- изделие p1 — 30;
- изделие p2 — 50;

Нужно определить, сколько изделий каждого типа следует произвести, чтобы получить максимальную прибыль.

In [13]:
p1_count, p2_count = Ints('p1_count p2_count')

b1_work_time, b2_work_time = Ints('b1_work_time b2_work_time')

max_work_time_b1 = IntVal(100)
max_work_time_b2 = IntVal(120)

profit_per_unit_p1 = IntVal(30)
profit_per_unit_p2 = IntVal(50)

opt = Optimize()

opt.add(p1_count * 2 + p2_count <= max_work_time_b1)
opt.add(p1_count + p2_count * 3 <= max_work_time_b2)

opt.add(And(p1_count >= 0, p2_count >= 0))

profit = p1_count * profit_per_unit_p1 + p2_count * profit_per_unit_p2

opt.maximize(profit)

opt.check()

sat

In [14]:
opt.model()

[p1_count = 36, p2_count = 28]

In [15]:
opt.model().evaluate(profit)

2480

#### Задача 2. Минимальная стоимость перевозки.

Есть два склада w1 и w2 и два магазина s1 и s2. 
Стоимость каждого маршрута:
- w1 -> s1 = 4;
- w1 -> s2 = 8;
- w2 -> s1 = 5;
- w2 -> s2 = 3;

Если известно, что:
- w1 поставляет ровно 80 товаров;
- w1 поставляет ровно 120 товаров;
- s1 получает 110 товаров;
- s2 получает 110 товаров;
  
Нужно выяснить с какого склада в какой магазин сколько отправить товаров чтобы общая стоимость была минимальна?

In [16]:
delivery_w1_to_s1, delivery_w1_to_s2 = Ints('delivery_w1_to_s1 delivery_w1_to_s2')
delivery_w2_to_s1, delivery_w2_to_s2 = Ints('delivery_w2_to_s1 delivery_w2_to_s2')

price_w1_s1 = IntVal(4)
price_w1_s2 = IntVal(8)
price_w2_s1 = IntVal(5)
price_w2_s2 = IntVal(3)

opt = Optimize()
opt.add(delivery_w1_to_s1 + delivery_w1_to_s2 == 100)
opt.add(delivery_w2_to_s1 + delivery_w2_to_s2 == 120)

opt.add(delivery_w1_to_s1 + delivery_w2_to_s1 == 110)
opt.add(delivery_w1_to_s2 + delivery_w2_to_s2 == 110)

opt.add(And(delivery_w1_to_s1 > 0, delivery_w1_to_s2 > 0, delivery_w2_to_s1 > 0, delivery_w2_to_s2 > 0))

cost = (price_w1_s1 * delivery_w1_to_s1 + price_w1_s2 * delivery_w1_to_s2 +
        price_w2_s1 * delivery_w2_to_s1 + price_w2_s2 * delivery_w2_to_s2)

opt.minimize(cost)

opt.check()

sat

In [17]:
opt.model()

[delivery_w2_to_s1 = 11,
 delivery_w1_to_s2 = 1,
 delivery_w2_to_s2 = 109,
 delivery_w1_to_s1 = 99]

In [18]:
opt.model().evaluate(cost)

786

### 5. Множественные солверы.

#### Задача 1. Проверка двух конфигураций сервера.

Нужно подорбать и сравнить несколько конфигураций сервиса.

Известны требования:
- оперативная память должна быть не менее 16 ГБ;
- процессор должен иметь не менее 16 ядер;
- объём SSD должен быть не менее 1000 ГБ;

Стоимость компонентов:
- 1 ГБ RAM — 2;
- 1 ядро CPU — 8;
- каждые 100 ГБ SSD — 5.

Общий бюджет не должен превышать 450.


In [19]:
BUDGET = 450

solver = Solver()

ram = Int("ram")
cpu = Int("cpu")
ssd = Int("ssd")

solver.add(ram >= 32, cpu >= 16, ssd >= 1000, ssd % 100 == 0,
           Implies(cpu > 24, ram >= 64),
           Implies(ssd > 2000, ram >= 64),
           ram * 2 + cpu * 8 + (ssd / 100) * 5 <= BUDGET)

solver.check()

sat

In [20]:
model = solver.model()

ram_val = model.evaluate(ram)
cpu_val = model.evaluate(cpu)
ssd_val = model.evaluate(ssd)

print(f"Config 1: {model}")

Config 1: [ssd = 2100, ram = 64, cpu = 25]


Явно укажем что альтернативная конфигурация не должна совпадать по парамтерам с уже рассчитанной.

In [21]:
solver_alternative = Solver()
solver_alternative.add(solver.assertions())

solver_alternative.add(And(ram != ram_val, cpu != cpu_val, ssd != ssd_val))

solver_alternative.check()

sat

In [22]:
model_alternative = solver_alternative.model()

print(f"Config 2: {model_alternative}")

Config 2: [ssd = 1000, ram = 65, cpu = 16]


#### Задача 2. Доказательство совместимости.

Убедться в совместимости фреймворка и провайдера бд.

Приложение

Для приложения действуют следующие ограничения:
- используется .NET 8;
- используется ASP.NET Core 8;
- используется EF Core 9;
- SQLite Provider;

Для SQLite Provider известны ограничения совместимости:
- версия провайдера должна совпадать с версией EF Core;
- версия провайдера не может быть выше версии .NET;
- если используется EF Core 9, то требуется .NET 9;

Выяснить совместимость провайдера и приложения:

In [47]:
app_solver = Solver()

dotnet = Int("dotnet")
aspnet = Int("aspnet")
ef = Int("ef")

app_solver.add(dotnet == 8, aspnet == 8, ef == 9)

app_solver.check()
app_model = app_solver.model()

app_model

[ef = 9, aspnet = 8, dotnet = 8]

In [48]:
sql_provider_solver = Solver()

sqlite = Int("sqlite")

sql_provider_solver.add(sqlite == app_model[ef],          
                        sqlite <= app_model[dotnet],      
                        Implies( sqlite == 9, app_model[dotnet] == 9))

is_compatibile = provider.check()

print(f'Provider compatibility: {is_compatibile}')

if is_compatibile == sat:
    print(provider.model())

Provider compatibility: unsat


Ожидаемо не совместим.